# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

My baseline will prioritize content items that have enough search visibility to evaluate and show observable signs of underperformance.

The rule will assign review points for the following signals:

1. Low CTR among pages with sufficient search impressions.
2. Low engagement rate among pages with measurable sessions.
3. Older content that has not been updated recently.
4. Weak average search position among pages with sufficient impressions.

Each signal will contribute one point to the total baseline score. Pages with more review signals will receive a higher priority.

This is a transparent, rule-based decision-support system. It does not predict Google's ranking algorithm or prove that updating a page will improve its performance.

### Reason codes

- `visible_low_ctr`: Sufficient impressions but relatively low CTR.
- `visible_low_engagement`: Measurable sessions but relatively low engagement.
- `stale_content`: Content has not been updated for at least 180 days.
- `weak_search_position`: Sufficient impressions but average position is weaker than the selected threshold.
- `multiple_issues`: Two or more review signals are present.
- `no_major_issue`: None of the above signals are present.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Locate the existing starter dataset
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-workspace2/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Starter CSV not found. Make sure the repository is cloned "
        "and data/raw/content_refresh_anonymized.csv is available."
    )

df = pd.read_csv(data_path)

# 2. Check required columns
required_columns = [
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "days_since_last_update",
    "avg_position",
]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# 3. Convert scoring columns to numeric safely
numeric_columns = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "days_since_last_update",
    "avg_position",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Define observable review signals
# The 300-impression floor follows the dataset's moderate impression tier.
visible = df["impressions_90d"] >= 300
measurable_sessions = df["sessions_90d"] > 0

# avg_position = 0 means no position data, not rank zero.
has_position = df["avg_position"] > 0

# Calculate low-performance thresholds from eligible rows.
ctr_cutoff = df.loc[visible, "ctr"].quantile(0.25)
engagement_cutoff = df.loc[measurable_sessions, "engagement_rate"].quantile(0.25)

df["flag_low_ctr"] = (
    visible & (df["ctr"] <= ctr_cutoff)
)

df["flag_low_engagement"] = (
    measurable_sessions &
    (df["engagement_rate"] <= engagement_cutoff)
)

df["flag_stale_content"] = (
    df["days_since_last_update"] >= 180
)

df["flag_weak_position"] = (
    visible & has_position & (df["avg_position"] > 20)
)

# 5. Calculate transparent baseline score
flag_columns = [
    "flag_low_ctr",
    "flag_low_engagement",
    "flag_stale_content",
    "flag_weak_position",
]

df["baseline_score"] = df[flag_columns].sum(axis=1)

# 6. Assign reason codes
def get_reason_codes(row):
    reasons = []

    if row["flag_low_ctr"]:
        reasons.append("visible_low_ctr")

    if row["flag_low_engagement"]:
        reasons.append("visible_low_engagement")

    if row["flag_stale_content"]:
        reasons.append("stale_content")

    if row["flag_weak_position"]:
        reasons.append("weak_search_position")

    if len(reasons) >= 2:
        reasons.append("multiple_issues")

    if not reasons:
        reasons.append("no_major_issue")

    return "; ".join(reasons)

df["reason_code"] = df.apply(get_reason_codes, axis=1)

# 7. Assign suggested editorial review action
def get_action(score):
    if score >= 3:
        return "Priority editorial review"
    elif score == 2:
        return "Review performance and content"
    elif score == 1:
        return "Monitor and investigate"
    return "No immediate action"

df["recommended_action"] = df["baseline_score"].apply(get_action)

# 8. Add confidence notes based on impression volume.
# These are review-confidence descriptions, not statistical probabilities.
def get_confidence(impressions):
    if impressions >= 3000:
        return "Higher visibility evidence"
    elif impressions >= 300:
        return "Moderate visibility evidence"
    return "Limited visibility evidence"

df["confidence_note"] = df["impressions_90d"].apply(get_confidence)

df["what_could_make_this_wrong"] = (
    "Low CTR or engagement may reflect query intent, "
    "content type, seasonality, or measurement limitations."
)

# 9. Rank all content items
df = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["baseline_rank"] = np.arange(1, len(df) + 1)

# 10. Save ranked queue
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_columns = [
    "baseline_rank",
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "days_since_last_update",
    "avg_position",
    "baseline_score",
    "reason_code",
    "recommended_action",
    "confidence_note",
    "what_could_make_this_wrong",
]

ranked = df[output_columns].copy()

output_path = output_dir / "baseline_action_score.csv"
ranked.to_csv(output_path, index=False)

print("Dataset shape:", df.shape)
print("CTR lower-quartile threshold:", round(ctr_cutoff, 3))
print("Engagement lower-quartile threshold:", round(engagement_cutoff, 3))
print("Total ranked content items:", len(ranked))
print("Saved file:", output_path)
print("\nBaseline score distribution:")
print(df["baseline_score"].value_counts().sort_index())
print("\nTop 10 ranked items:")
display(ranked.head(10))


FileNotFoundError: Starter CSV not found. Make sure the repository is cloned and data/raw/content_refresh_anonymized.csv is available.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.